# Gemini 2.5 Flash — Inference with the OpenAI SDK

## Setup

In [ ]:
%pip install openai -q

In [ ]:
from openai import OpenAI

API_KEY = ""
MODEL   = "gemini-2.5-flash"

client = OpenAI(
    api_key  = API_KEY,
    base_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
    
)

---
## 1 — Q&A

In [8]:
question = "What is the difference between supervised and unsupervised learning?"

response = client.chat.completions.create(
    model    = MODEL,
    messages = [
        {"role": "user", "content": question}
    ],
    max_tokens=1024,
    temperature=0
)

print(response.choices[0].message.content)

The fundamental difference between supervised and unsupervised learning lies in the **nature of the data they are trained on** and, consequently, their **goals**.

Here's a breakdown:

---

##


In [ ]:
# Try a different question
question2 = "Explain what a transformer architecture is in simple terms."

response2 = client.chat.completions.create(
    model    = MODEL,
    messages = [
        {"role": "user", "content": question2}
    ]
)

print(response2.choices[0].message.content)

---
## 2 — System Prompt

In [ ]:
# The system message sets the model's persona and behavior for the entire conversation.
# Here we instruct it to act as a strict Python tutor that only answers Python questions.

system_prompt = """
You are a Python programming tutor for university students.
You only answer questions related to Python.
If the user asks about anything else, politely decline and redirect them to Python.
Keep answers concise and always include a short code example.
"""

user_question = "How do list comprehensions work?"

response = client.chat.completions.create(
    model    = MODEL,
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": user_question}
    ]
)

print(response.choices[0].message.content)

In [ ]:
# Watch what happens when the user goes off-topic
off_topic = "Can you explain the French Revolution?"

response_off = client.chat.completions.create(
    model    = MODEL,
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": off_topic}
    ]
)

print(response_off.choices[0].message.content)

---
## 3 — Multi-Turn Chat

In [ ]:
# In a multi-turn conversation the full history is sent with every request.
# The model has no memory of its own — we maintain the context ourselves.

def chat(history, user_message):
    history.append({"role": "user", "content": user_message})
    response = client.chat.completions.create(
        model    = MODEL,
        messages = history
    )
    assistant_reply = response.choices[0].message.content
    history.append({"role": "assistant", "content": assistant_reply})
    return assistant_reply, history

In [ ]:
history = []

reply, history = chat(history, "Hi! I want to learn about neural networks. Where should I start?")
print("Turn 1:\n", reply, "\n")

reply, history = chat(history, "What is backpropagation exactly?")
print("Turn 2:\n", reply, "\n")

reply, history = chat(history, "Can you give me a very simple analogy for it?")
print("Turn 3:\n", reply)

In [ ]:
# Inspect the full history — this is exactly what gets sent to the API each time
for msg in history:
    print(f"[{msg['role'].upper()}]")
    print(msg['content'][:200], "..." if len(msg['content']) > 200 else "")
    print()

---
## 4 — Summarization

In [ ]:
long_text = """
Large language models (LLMs) are a type of artificial intelligence model trained on vast amounts of
text data. They use a transformer architecture with billions of parameters, allowing them to understand
and generate human-like text. Training involves predicting the next token in a sequence, which forces
the model to learn grammar, facts, reasoning, and even some degree of common sense from the data.

After pretraining, models are typically fine-tuned using techniques such as Reinforcement Learning
from Human Feedback (RLHF), which aligns the model's outputs with human preferences and makes it
more helpful and less harmful. Popular LLMs include GPT-4, Claude, Gemini, and LLaMA.

Despite their capabilities, LLMs have notable limitations: they can hallucinate facts, lack true
understanding, have a training knowledge cutoff, and can reflect biases present in their training data.
Research into improving reliability, reducing hallucinations, and enabling longer context windows
is ongoing.
"""

response = client.chat.completions.create(
    model    = MODEL,
    messages = [
        {"role": "system", "content": "Summarize the given text in 3 bullet points. Be concise."},
        {"role": "user",   "content": long_text}
    ]
)

print(response.choices[0].message.content)

In [ ]:
# Same text, different instruction — summarize for a 10-year-old
response_simple = client.chat.completions.create(
    model    = MODEL,
    messages = [
        {"role": "system", "content": "Summarize the given text as if explaining to a 10-year-old. Use simple words."},
        {"role": "user",   "content": long_text}
    ]
)

print(response_simple.choices[0].message.content)